# Earth-like

This notebook runs the shipped `earth-slab` configuration for 30 days -- SPEEDY
T31L8 with realistic orography and a climatological surface: a relaxed slab
ocean, a slab sea-ice model, and SPEEDY's own slab land, which jax-gcm
integrates inside the atmosphere -- and maps the final sea surface temperature
and the land temperature anomaly. On a laptop CPU it takes about 2 minutes. This run is one command:

```bash
python -m jem.main +configuration=earth-slab
```

The terrain, the climatological forcing, the 30-day ocean relaxation and the sea ice seeded from the observed concentration are all set -- with the reason for each -- in `earth-slab.yaml`. This notebook builds the same configuration with `jem.configurations.load`, which returns the coupled model built from a `jem/config/configuration/*.yaml` as plain Python objects (`exp.coupler`, `exp.config`, `exp.run_kwargs`), and reads the tuned values back out of `exp.config` rather than copying them into Python, where they could drift from the file.

## Build it

In [ ]:
from pathlib import Path

from jem import configurations, plot, run_chunked

output_dir = (Path("output") / "02-01_earth").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
exp = configurations.load("earth-slab")
exp.coupler

The tuned settings `earth-slab.yaml` documents, read back from `exp.config` rather than restated here (the sea-ice climatology is shown by file name, since its directory depends on where JAX-ESM is installed):

In [ ]:
{
    "ocean.params.relaxation_time [days]":
        exp.config["ocean"]["params"]["relaxation_time"] / 86400,
    "seaice.ice_clim_file": Path(exp.config["seaice"]["ice_clim_file"]).name,
}

## Run it

In [ ]:
result = run_chunked(
    exp.coupler,
    **{
        **exp.run_kwargs,
        "output_dir": str(output_dir),
        "subsample": 3,       # 10 records out of 30 coupled days
        "checkpoint_path": None,
    },
)
result.steps_completed, [p.name for p in result.paths]

## What it wrote

One file per component per chunk, named after the coupled step its chunk starts on (`<component>-<first step>.nc`).

The land is the atmosphere's: its slab-land temperature is written with the atmosphere's output, as `land_model.stl_anom`, the anomaly from the `stl` climatology the run is forced with.

In [ ]:
atm_ds = plot.open_output(output_dir, "atm")
ocn_ds = plot.open_output(output_dir, "ocn")
sorted(name for name in atm_ds.data_vars if name.startswith("land_model."))

## Plot

The ocean map shows only the cells that contain sea: the slab ocean is defined on the atmosphere's whole grid, and a cell that is entirely land carries a constant placeholder temperature (the 288.15 K initial value) that the ocean never updates, so it is masked with the ocean grid's `ocean_mask` rather than drawn as a 15 °C continent.

The land map is a snapshot of a spin-up from rest, not a climatology: the run starts from an isothermal atmosphere with no wind, and the slab land is still adjusting to that, so the anomalies from the `stl` climatology can be tens of kelvin. Its range is printed below the map.

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

fig, axes = plt.subplots(
    1, 2, figsize=(13, 5), subplot_kw={"projection": ccrs.PlateCarree()}
)

# The ocean's own mask (True where a cell contains any sea): the same one the
# ocean integrates with, so what is drawn is exactly what was computed.
sst = ocn_ds["sea_surface_temperature"].isel(time=-1) - 273.15
ocean = xr.DataArray(
    np.asarray(exp.coupler.components["ocn"].grid.ocean_mask, dtype=bool),
    dims=sst.dims,
)
plot.map_plot(sst.where(ocean), ax=axes[0], coastlines=True,
              title="Sea surface temperature [°C] (ocean cells only)")

land_anomaly = atm_ds["land_model.stl_anom"].isel(time=-1)
plot.map_plot(land_anomaly, ax=axes[1], coastlines=True,
              title="Slab-land temperature anomaly from climatology [K]")
plt.tight_layout()

print(f"land anomaly range [K]: {float(land_anomaly.min()):.1f} to "
      f"{float(land_anomaly.max()):.1f}")